# Simulation Step (PMDCo / OBI): from data entry to RDF

This notebook shows how to describe a simulation step and convert that
description into a standardised, machine-readable RDF graph following the
[OBI](http://purl.obolibrary.org/obo/obi.owl) and
[PMDCo](https://w3id.org/pmd/co/) ontologies.

**You only need to edit one file:** `docs/example.input.json`. Everything
else is automatic.

A simulation step in this schema captures:

- a **name** and optional **description**
- the **input datasets or material cards** it consumes
- the **output datasets or result files** it produces
- which **steps precede** it in the analysis chain
- optional **quantitative simulation parameters** (solver, mesh size, time step, etc.)


## What the notebook does

```
example.input.json
  |  your simulation name, inputs, outputs, and parameters
  |
  v
Transform
  |  converts your plain JSON into a structured OO-LD document
  |
  v
RDF graph
  |  machine-readable, ontology-aligned triples
  |
  v
SHACL validation  ->  confirms the graph is structurally correct
SPARQL inspect    ->  shows the key connections recorded in the graph
```


## Environment setup

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install jupyterlab
jupyter lab
```

In [1]:
%pip install -q jsonata-python rdflib pyshacl pyyaml

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, json, pathlib, yaml, pyshacl, rdflib
from jsonata.jsonata import Jsonata
from importlib.metadata import version

print("Python        :", sys.executable)
print("jsonata-python", version("jsonata-python"))
print("rdflib        ", version("rdflib"))
print("pyshacl       ", version("pyshacl"))

HERE   = pathlib.Path().resolve()   # docs/
SCHEMA = HERE.parent                # simulation/step/PMDCo/

Python        : /root/semantic-dataspace/.venv/bin/python3
jsonata-python 0.6.2
rdflib         7.6.0
pyshacl        0.31.0


## Step 1: Describe your simulation

Edit `docs/example.input.json` with your data, then run this cell to load it.

| Field | Required | Description |
|---|---|---|
| `step_name` | yes | A name for this simulation step |
| `description` | no | Free-text explanation of what the simulation does |
| `inputs` | no | List of IRIs of datasets or material cards consumed |
| `outputs` | no | List of IRIs of result datasets produced |
| `preceded_by` | no | List of IRIs of steps that directly precede this one |
| `conditions` | no | List of simulation parameters (name, value, unit) |
| `step_id` | no | Custom IRI slug; auto-derived from `step_name` if omitted |

**What is an IRI?**  
An IRI (Internationalized Resource Identifier) is a web address used to uniquely
identify a thing in a knowledge graph. For input and output datasets, use the
IRI that was assigned when those datasets were registered. For example:
`https://example.org/material-cards/316L-hockett-sherby-v1`.  
If datasets do not yet have IRIs, leave `inputs` and `outputs` empty for now
and add them later.

In [3]:
simplified = json.loads((HERE / "example.input.json").read_text())

print(json.dumps(simplified, indent=2))

{
  "step_name": "FEM stress analysis run 1",
  "description": "Implicit static FEM analysis of tensile specimen geometry using calibrated Hockett-Sherby material card.",
  "inputs": [
    "https://example.org/material-cards/316L-hockett-sherby-v1"
  ],
  "outputs": [
    "https://example.org/simulation-results/fem-stress-316L-run-1"
  ],
  "preceded_by": [
    "https://example.org/simulations/model-calibration-316L-batch-1"
  ],
  "conditions": [
    {
      "name": "Solver",
      "unit": "Abaqus/Standard 2023"
    },
    {
      "name": "Mesh Size",
      "value": 0.5,
      "unit": "mm"
    },
    {
      "name": "Load Step",
      "value": 0.01,
      "unit": "s"
    }
  ]
}


## Step 2: Convert to OO-LD

This step transforms your plain JSON into a structured
[OO-LD](https://github.com/OO-LD/oold-python) document.
OO-LD is standard JSON enriched with ontology mappings: every field name
is linked to a precise ontology term, so machines can interpret it unambiguously.

The converter also assigns stable identifiers to each node in the graph.
By default, the step identifier is derived from the step name
(e.g. `"FEM stress analysis run 1"` becomes `simulation-fem-stress-analysis-run-1`).
You can override this with a custom `step_id` in the input file.

You can also run the transform from the command line:
```
python -m jsonata -e ../simplified/transform.jsonata -i example.input.json
```

In [4]:
expr     = (SCHEMA / "simplified" / "transform.jsonata").read_text()
oold_doc = Jsonata(expr).evaluate(simplified)

print(json.dumps(oold_doc, indent=2))

{
  "conforms_to": "https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/simulation/step/PMDCo/#v1.0.0",
  "type": "obi:0000471",
  "id": "simulation-fem-stress-analysis-run-1",
  "label": "FEM stress analysis run 1",
  "description": "Implicit static FEM analysis of tensile specimen geometry using calibrated Hockett-Sherby material card.",
  "has_specified_input": [
    "https://example.org/material-cards/316L-hockett-sherby-v1"
  ],
  "has_specified_output": [
    "https://example.org/simulation-results/fem-stress-316L-run-1"
  ],
  "preceded_by": [
    "https://example.org/simulations/model-calibration-316L-batch-1"
  ],
  "has_process_condition": [
    {
      "type": "pmdco:PMD_0000013",
      "id": "simulation-fem-stress-analysis-run-1-param-1",
      "parameter_label": "Solver",
      "parameter_unit": "Abaqus/Standard 2023"
    },
    {
      "type": "pmdco:PMD_0000013",
      "id": "simulation-fem-stress-analysis-run-1-param-2",
      "parameter_label": "Me

## Step 3: Convert to RDF

The OO-LD document is now parsed into an RDF graph.
RDF (Resource Description Framework) is the standard format for knowledge graphs:
everything is expressed as triples of the form *subject, predicate, object*.

The ontology context from `specs/schema.oold.yaml` tells the parser which
ontology IRI each field maps to. For example, `has_specified_input` becomes
`http://purl.obolibrary.org/obo/OBI_0000293`, the canonical OBI term for
the input of a planned process.

In [5]:
context = yaml.safe_load((SCHEMA / "specs" / "schema.oold.yaml").read_text())["@context"]

g = rdflib.Dataset()
g.parse(data=json.dumps({"@context": context, **oold_doc}), format="json-ld")

print(f"Graph contains {len(g)} triples.\n")
print(g.serialize(format="turtle"))

Graph contains 21 triples.

@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix ns1: <http://purl.obolibrary.org/obo/> .
@prefix pmdco: <https://w3id.org/pmd/co/> .
@prefix qudt: <http://qudt.org/schema/qudt/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<https://w3id.org/pmd/co/test/simulation-fem-stress-analysis-run-1> a ns1:OBI_0000471 ;
    rdfs:label "FEM stress analysis run 1" ;
    ns1:BFO_0000062 <https://example.org/simulations/model-calibration-316L-batch-1> ;
    ns1:OBI_0000293 <https://example.org/material-cards/316L-hockett-sherby-v1> ;
    ns1:OBI_0000299 <https://example.org/simulation-results/fem-stress-316L-run-1> ;
    dcterms:conformsTo <https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/simulation/step/PMDCo/#v1.0.0> ;
    rdfs:comment "Implicit static FEM analysis of tensile specimen geometry using calibrated Hockett-Sherby material card." ;
    pmdco:PMD_0000016 <https://w3i

In [6]:
# Optional: save to file
out_ttl = HERE / "output_simulation_step.ttl"
out_ttl.write_text(g.serialize(format="turtle"))
print(f"Written to {out_ttl}")

Written to /root/semantic-dataspace/semantic-schemas/schemas/simulation/step/PMDCo/docs/output_simulation_step.ttl


## Step 4: Validate against the SHACL shape

SHACL (Shapes Constraint Language) defines rules that a valid RDF graph must
satisfy. The shape file `specs/shape.ttl` checks that:

- The simulation step has exactly one label.
- Every input and output is recorded as an IRI (not a plain text string).
- Every simulation parameter has a label, and its value (if given) is a number.

In [7]:
shapes_graph = rdflib.Graph().parse(str(SCHEMA / "specs" / "shape.ttl"))

conforms, results_graph, _ = pyshacl.validate(
    g,
    shacl_graph = shapes_graph,
)

print(f"Conforms: {conforms}")
if not conforms:
    SH = rdflib.Namespace("http://www.w3.org/ns/shacl#")
    for result in results_graph.subjects(rdflib.RDF.type, SH.ValidationResult):
        msg  = results_graph.value(result, SH.resultMessage)
        path = results_graph.value(result, SH.resultPath)
        loc  = str(path).rsplit("#", 1)[-1].rsplit("/", 1)[-1] if path else None
        print(f"\n  x {msg}" + (f"\n    property: {loc}" if loc else ""))

Conforms: True


## Step 5: Inspect the graph

The cells below use SPARQL (a query language for RDF graphs) to confirm that
the key facts from your input were recorded correctly.

You do not need to understand SPARQL to use this notebook. Just run the cells
and check that the output matches what you entered.

In [8]:
# Flatten the Dataset into a plain Graph for SPARQL queries
flat = rdflib.Graph()
for s, p, o, _ in g.quads():
    flat.add((s, p, o))

OBI = rdflib.Namespace("http://purl.obolibrary.org/obo/OBI_")

# Print step label
step_iri = next(flat.subjects(rdflib.RDF.type, OBI["0000471"]))
label    = flat.value(step_iri, rdflib.RDFS.label)
print(f"Simulation step : {step_iri}")
print(f"Label           : {label}")

Simulation step : https://w3id.org/pmd/co/test/simulation-fem-stress-analysis-run-1
Label           : FEM stress analysis run 1


In [9]:
SPARQL_IO = """
PREFIX obi:  <http://purl.obolibrary.org/obo/OBI_>
PREFIX bfo:  <http://purl.obolibrary.org/obo/BFO_>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?role ?iri
WHERE {
  { ?step a obi:0000471 ; obi:0000293 ?iri . BIND("input"       AS ?role) }
  UNION
  { ?step a obi:0000471 ; obi:0000299 ?iri . BIND("output"      AS ?role) }
  UNION
  { ?step a obi:0000471 ; bfo:0000062 ?iri . BIND("preceded by" AS ?role) }
}
ORDER BY ?role ?iri
"""

rows = list(flat.query(SPARQL_IO))
if rows:
    print(f"{'Role':<14}  IRI")
    print("-" * 70)
    for r in rows:
        print(f"{str(r.role):<14}  {r.iri}")
else:
    print("No inputs, outputs, or predecessors recorded.")

Role            IRI
----------------------------------------------------------------------
input           https://example.org/material-cards/316L-hockett-sherby-v1
output          https://example.org/simulation-results/fem-stress-316L-run-1
preceded by     https://example.org/simulations/model-calibration-316L-batch-1


In [10]:
SPARQL_COND = """
PREFIX obi:  <http://purl.obolibrary.org/obo/OBI_>
PREFIX pmd:  <https://w3id.org/pmd/co/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX qudt: <http://qudt.org/schema/qudt/>

SELECT ?name ?value ?unit
WHERE {
  ?step a obi:0000471 ;
        pmd:PMD_0000016 ?cond .
  ?cond rdfs:label ?name .
  OPTIONAL { ?cond qudt:value ?value . }
  OPTIONAL { ?cond qudt:hasUnit ?unit . }
}
ORDER BY ?name
"""

rows = list(flat.query(SPARQL_COND))
if rows:
    print(f"{'Parameter':<30}  {'Value':>10}  Unit")
    print("-" * 60)
    for r in rows:
        val  = f"{float(r.value):>10.3f}" if r.value else f"{'n/a':>10}"
        unit = str(r.unit) if r.unit else ""
        print(f"{str(r.name):<30}  {val}  {unit}")
else:
    print("No simulation parameters recorded.")

Parameter                            Value  Unit
------------------------------------------------------------
Load Step                            0.010  s
Mesh Size                            0.500  mm
Solver                                 n/a  Abaqus/Standard 2023


## Summary

| Step | What happens |
|---|---|
| 1 | You fill in `example.input.json` with the simulation name, datasets, and parameters |
| 2 | The data is converted to an OO-LD document (ontology-mapped JSON) |
| 3 | The OO-LD is parsed into an RDF graph |
| 4 | The graph is validated against the SHACL shape |
| 5 | Key facts are extracted from the graph to confirm correctness |

To describe a different simulation, edit `docs/example.input.json` and re-run all cells.


## Further reading

- [OO-LD primer](../../../docs/oold-primer.md): what OO-LD is and how it works
- [Schema format reference](../../../docs/schema-format.md): how to write your own schema
- [Schema patterns](../../../docs/schema-patterns.md): inheritance and composition patterns
- [Simplified input guide](../../../docs/simplified-input-guide.md): end-to-end workflow
- See `simulation/model-calibration/PMDCo/` for an extended schema that specialises this one
- See `workflow/PMDCo/docs/workflow_demo.ipynb` for a cross-domain demo that uses this schema